# From two target timelines to a cross-attention read

Prerequisite: the Python primer and Chapters 13–14. All evidence is also printed in the chapters. Encoder states are computed; decoder conditional rows and attention queries are supplied diagnostic values. No decoder training is claimed. Code: Apache-2.0; prose: CC BY-SA 4.0.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'code/part-iii').is_dir():
    root = next(p for p in root.parents if (p / 'code/part-iii').is_dir())
sys.path.insert(0, str(root / 'code/part-iii'))
from core import fixture, results
import json
import math
from fractions import Fraction
data = fixture()
observed = results(data)
print('Fixture:', data['version'], '| CPU | deterministic supplied values')

Fixture: part-iii-sequence-v1 | CPU | deterministic supplied values


## Compute the source, then compare the two clocks

The encoder consumes three source tokens. The target contains four predictions including EOS. Locate the first source-state collision, then the first different decoder input. These are separate events.

In [2]:
for red, blue in zip(observed['encoder'], observed['contrast_encoder']):
    print('source step', red['step'], 'red:', red['state'], 'blue:', blue['state'])
print('teacher:', json.dumps(observed['teacher']))
print('generation:', json.dumps(observed['generation']))
assert observed['teacher'][1]['input'] == 'red'
assert observed['generation']['trace'][1]['input'] == 'blue'
print('reference probability:', observed['target_sequence_probability'], 'mean NLL:', observed['target_mean_nll'])

source step 1 red: [0.6931471805599453, 0.0] blue: [0.0, 0.0]
source step 2 red: [0.0, 0.6931471805599453] blue: [0.0, 0.6931471805599453]
source step 3 red: [0.0, 0.0] blue: [0.0, 0.0]
teacher: [{"step": 1, "input": "BOS", "target": "red", "target_probability": 0.4, "greedy": "blue"}, {"step": 2, "input": "red", "target": "key", "target_probability": 0.8, "greedy": "key"}, {"step": 3, "input": "key", "target": ".", "target_probability": 0.9, "greedy": "."}, {"step": 4, "input": ".", "target": "EOS", "target_probability": 1, "greedy": "EOS"}]
generation: {"trace": [{"step": 1, "input": "BOS", "output": "blue", "probability": 0.5}, {"step": 2, "input": "blue", "output": "door", "probability": 0.7}, {"step": 3, "input": "door", "output": ".", "probability": 0.9}, {"step": 4, "input": ".", "output": "EOS", "probability": 1}], "stopped_at_eos": true}
reference probability: 0.2880000000000001 mean NLL: 0.3111986997115477


## Two queries, unchanged memory

Independent exact expectations follow from exp(ln(2)) = 2: exponentials [2,1,1] normalize to [1/2,1/4,1/4]. Exchanging query coordinates exchanges the first two weights. The context averages source states, not output-word probabilities.

In [3]:
for query, result in zip(data['attention']['queries'], observed['attention']):
    print('query:', query, json.dumps(result))
assert observed['attention'][0]['weights'] == [.5, .25, .25]
assert observed['attention'][1]['weights'] == [.25, .5, .25]

query: [1, 0] {"kept_positions": [0, 1, 2], "scores_on_kept_positions": [0.6931471805599453, 0.0, 0.0], "weights": [0.5, 0.25, 0.25], "context": [0.34657359027997264, 0.17328679513998632]}
query: [0, 1] {"kept_positions": [0, 1, 2], "scores_on_kept_positions": [0.0, 0.6931471805599453, 0.0], "weights": [0.25, 0.5, 0.25], "context": [0.17328679513998632, 0.34657359027997264]}


## State removal is not source-token removal

Hold the query and scalar readout fixed. Removing the stored key state gives context [2a/3,0]. Removing the original key token changes the later punctuation state, giving context [a,0]. Highest attention weight alone does not determine the largest effect on the declared readout.

In [4]:
for slot, source in zip(observed['slot_deletions'], observed['source_deletions']):
    print('removed:', slot['removed_position'], 'slot context:', slot['context'], 'slot readout:', slot['readout'], 'reencoded context:', source['context'], 'reencoded readout:', source['readout'])
assert math.isclose(observed['slot_deletions'][0]['readout'], math.log(2))
assert observed['slot_deletions'][1]['context'] != observed['source_deletions'][1]['context']

removed: 0 slot context: [0.0, 0.34657359027997264] slot readout: 0.6931471805599453 reencoded context: [0.0, 0.34657359027997264] reencoded readout: 0.6931471805599453
removed: 1 slot context: [0.46209812037329684, 0.0] slot readout: 0.46209812037329684 reencoded context: [0.6931471805599453, 0.0] reencoded readout: 0.6931471805599453
removed: 2 slot context: [0.46209812037329684, 0.23104906018664842] slot readout: 0.9241962407465937 reencoded context: [0.46209812037329684, 0.23104906018664842] reencoded readout: 0.9241962407465937


## Optional NumPy recomputation

This independent matrix path uses NumPy 2.0.2 in the declared CPU environment. It repeats the same unscaled score and row-wise source normalization. The preceding standard-library cells already establish the core mechanism.

In [5]:
import numpy as np
memory = np.array([row['state'] for row in observed['encoder']], dtype=np.float64)
queries = np.array(data['attention']['queries'], dtype=np.float64)
scores = queries @ memory.T
positive = np.exp(scores - scores.max(axis=1, keepdims=True))
weights = positive / positive.sum(axis=1, keepdims=True)
contexts = weights @ memory
np.testing.assert_allclose(weights, [[.5,.25,.25],[.25,.5,.25]], rtol=0, atol=1e-14)
np.testing.assert_allclose(contexts, np.log(2) * np.array([[.5,.25],[.25,.5]]), rtol=0, atol=1e-14)
print('NumPy', np.__version__, 'scores:', scores, 'weights:', weights, 'contexts:', contexts, sep='\n')

NumPy
2.0.2
scores:
[[0.69314718 0.         0.        ]
 [0.         0.69314718 0.        ]]
weights:
[[0.5  0.25 0.25]
 [0.25 0.5  0.25]]
contexts:
[[0.34657359 0.1732868 ]
 [0.1732868  0.34657359]]


## Transfer query and empty memory boundary

For query [1,1], the exponentials [2,2,1] give context [2a/5,2a/5]. Excluding all positions has no normalized read. The program must reject that operation instead of pretending a zero vector of weights is a probability distribution.

In [6]:
from core import attend
result = attend([1, 1], memory.tolist())
print('transfer query:', result)
assert all(math.isclose(x, .4 * math.log(2)) for x in result['context'])
try:
    attend([1, 0], memory.tolist(), [])
except ValueError as error:
    print('expected rejection:', str(error))
else:
    raise AssertionError('empty memory was accepted')

transfer query: {'kept_positions': [0, 1, 2], 'scores_on_kept_positions': [0.6931471805599453, 0.6931471805599453, 0.0], 'weights': [0.4, 0.4, 0.2], 'context': [0.2772588722239781, 0.2772588722239781]}
expected rejection: at least one distinct valid memory position required
